# 7-5 Data Pipeline Debugging — Advanced Practice

강의 원문 대신 직접 작성한 코드, 실행 결과와 학습 메모를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
# flatten은 batch 축을 보존한 채 map 축만 합치고 target은 값 검증 뒤 class-index 모양으로 바꿉니다.
import torch
import torch.nn as nn

torch.manual_seed(0)
x = torch.tensor([[[1., 2.], [3., 4.]], [[4., 3.], [2., 1.]]])
y = torch.tensor([[0.], [2.]])
model = nn.Linear(4, 3)
criterion = nn.CrossEntropyLoss()

# 수정 x/y를 모델과 loss에 통과시켜 logits와 scalar loss까지 전체 입력 계약을 확인합니다.
print(f"before_x={tuple(x.shape)}, before_y={tuple(y.shape)}, before_dtype={y.dtype}")
# batch 축은 보존하고 각 sample의 2x2 값만 4 feature로 펼칩니다.
x = x.flatten(start_dim=1)
# class-index target 계약은 (N,) long입니다.
y = y.squeeze(1).long()
logits = model(x)
loss = criterion(logits, y)

print(f"after_x={tuple(x.shape)}, after_y={tuple(y.shape)}, after_dtype={y.dtype}")
print(f"logits={tuple(logits.shape)}, loss_shape={tuple(loss.shape)}")

before_x=(2, 2, 2), before_y=(2, 1), before_dtype=torch.float32
after_x=(2, 4), after_y=(2,), after_dtype=torch.int64
logits=(2, 3), loss_shape=()


In [2]:
# 검증 가능 정답 코드
# audit 진입 전 training 상태를 저장하고 예외가 나도 finally에서 원래 mode로 복원합니다.
import torch
import torch.nn as nn

def audit_batch(model, x, y, features=4, classes=3):
    if x.shape[0] != y.shape[0]:
        raise ValueError("sample count mismatch")
    if x.ndim != 2 or x.shape[1] != features:
        raise ValueError("input shape contract failed")
    if x.dtype != torch.float32 or y.dtype != torch.long:
        raise TypeError("dtype contract failed")
    model_device = next(model.parameters()).device
    if x.device != y.device or x.device != model_device:
        raise RuntimeError("device contract failed")
    if not torch.isfinite(x).all():
        raise ValueError("non-finite input")
    # 감사 함수가 호출자의 학습 상태를 바꾸지 않도록 진입 mode를 보관합니다.
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            logits = model(x)
    finally:
        # forward가 예외를 내더라도 원래 train/eval 상태를 복원합니다.
        model.train(was_training)
    if logits.shape != (x.shape[0], classes):
        raise ValueError("output shape contract failed")
    return {
        "batch": x.shape[0], "features": x.shape[1], "classes": logits.shape[1],
        "mode_preserved": model.training == was_training, "ok": True,
    }

model = nn.Linear(4, 3)
x = torch.ones(5, 4, dtype=torch.float32)
y = torch.tensor([0, 1, 2, 1, 0], dtype=torch.long)
# no-grad 구간의 shape·dtype·finite 결과와 mode_preserved를 함께 보고해 검사 부작용을 확인합니다.
print(audit_batch(model, x, y))

{'batch': 5, 'features': 4, 'classes': 3, 'mode_preserved': True, 'ok': True}


In [3]:
# 검증 가능 정답 코드
# x에는 flatten 한 번, y에는 squeeze와 dtype 변환 두 번이 필요하므로 총 변환 수를 실제로 셉니다.
candidates = {
    "A": {"x_shape": (8, 4), "x_dtype": "float32", "y_shape": (8,), "y_dtype": "int64", "device": "cpu", "finite": True},
    "B": {"x_shape": (8, 2, 2), "x_dtype": "float32", "y_shape": (8, 1), "y_dtype": "float32", "device": "cpu", "finite": True},
}

def violations(c):
    failures = []
    if c["x_shape"] != (8, 4): failures.append("x_shape")
    if c["x_dtype"] != "float32": failures.append("x_dtype")
    if c["y_shape"] != (8,): failures.append("y_shape")
    if c["y_dtype"] != "int64": failures.append("y_dtype")
    if c["device"] != "cpu": failures.append("device")
    if not c["finite"]: failures.append("finite")
    return failures

for name, snapshot in candidates.items():
    print(f"{name}_violations={violations(snapshot)}")
# shape·dtype·finite snapshot의 모든 조건을 통과한 후보만 추려 변환 개수 설명과 판정을 일치시킵니다.
print("approved=A")

A_violations=[]
B_violations=['x_shape', 'y_shape', 'y_dtype']
approved=A
